# Sample 04: 手続き的ライター (`BinaryWriter`), バリアント & 仕様書直接出力

ストリーム指向の手続き型書き込み、各種文字列エンコーディング、多態バリアント (`write_variant`)、繰り返しチャンク集約、および仕様書 (Markdown) & C言語ヘッダーの直接出力を学びます。

### 学べる内容
- `BinaryWriter` による命令的バイナリ出力
- 文字列戦略: Null終端 (`write_cstring`), 長さプレフィックス (`write_prefixed_string`), 固定長 (`write_fixed_string`)
- 多態バリアント書き込み (`write_variant`) と候補検証
- 繰り返しレコード集約 (`with writer.set_caption(..., spec_count=...)`)
- `writer.write_markdown()` による仕様書直接エクスポート
- `writer.to_c_header()` による C言語ヘッダー直接エクスポート
- `BinaryReader` によるストリーム読み込み復元
- 注釈付き Hexdump とフィールドトレーステーブル (`writer.dump('table')`)

In [1]:
from pathlib import Path

from binary_master import (
    BinaryReader,
    BinaryWriter,
    FixedArray,
    Float32,
    UInt8,
    UInt16,
    UInt32,
    binary_struct,
)

## 1. 構造体の定義とライターの初期化

In [2]:
@binary_struct
class HeaderChunk:
    protocol_version: UInt16
    flags: UInt16

@binary_struct
class TextChunk:
    length: UInt16
    content: FixedArray[UInt8, 16]

@binary_struct
class DataRecord:
    record_id: UInt32
    timestamp: UInt32
    value: Float32

writer = BinaryWriter(default_endian="little")

## 2. セクション定義と各種文字列の書き込み

`writer.set_caption()` で論理グループを宣言しながらバイナリを書き進めます。

In [3]:
# ファイルヘッダー
writer.set_caption("File Header", desc="ファイル識別子とバージョン")
writer.write_uint32(0x46494C45, name="magic", desc="Magic 'FILE'")
writer.write_uint16(2, name="version_major", desc="Major version")
writer.write_uint16(0, name="version_minor", desc="Minor version")

# 各種文字列
writer.set_caption("Metadata", desc="アプリケーション属性テキスト")
writer.write_cstring("SampleApp v2.0", encoding="utf-8", name="app_name", desc="Null終端文字列")
writer.write_prefixed_string("Confidential Document", prefix_bytes=2, encoding="utf-8", name="doc_title", desc="2B長さプレフィックス文字列")
writer.write_fixed_string("AUTH", length=8, pad_byte=b" ", encoding="utf-8", name="author_tag", desc="8B固定長文字列")

print(f"現在オフセット: {writer.tell()} バイト")

現在オフセット: 54 バイト

## 3. 多態バリアント (`write_variant`) と繰り返しレコード集約

タグ値 `chunk_type` に応じて異なる構造体を安全に書き込みます。
また、`with writer.set_caption(..., spec_count="num_records"):` で囲むことで、仕様書上では1行のテンプレートレコードに自動集約されます。

In [4]:
writer.set_caption("Dynamic Payload", desc="動的ペイロードブロック")
candidates = {
    1: HeaderChunk,
    2: TextChunk,
}
writer.write_uint16(1, name="chunk_type", desc="1=HeaderChunk, 2=TextChunk")
writer.write_variant(
    HeaderChunk(protocol_version=10, flags=0x0001),
    candidates=candidates,
    tag_field="chunk_type",
    name="payload",
    desc="Configuration header variant",
)

# 繰り返しレコード
records = [
    DataRecord(record_id=1, timestamp=1000, value=25.5),
    DataRecord(record_id=2, timestamp=1001, value=26.0),
    DataRecord(record_id=3, timestamp=1002, value=26.5),
]
writer.write_uint16(len(records), name="num_records", desc="レコード件数")
with writer.set_caption("DataRecord", desc="連続計測データレコード", spec_count="num_records"):
    for record in records:
        writer.write_struct(record)

raw_data = writer.to_bytes()
print(f"書き込み完了総サイズ: {len(raw_data)} バイト")

書き込み完了総サイズ: 98 バイト

## 4. 仕様書 (Markdown) & C言語ヘッダーの直接出力

`BinaryWriter` に書き込まれた実行ログから、Builder を介さずワンライナーで仕様書と C言語ヘッダーを直接出力できます！

In [5]:
sample_dir = Path("/home/ishii/PycharmProjects/binary_master/sample")
spec_path = sample_dir / "writer_output_spec.md"
header_path = sample_dir / "writer_output.h"

writer.write_markdown(spec_path, title="Procedural Binary Protocol Specification")
writer.write_c_header(header_path)

print(f"仕様書 Markdown 生成: {spec_path.name}")
print(f"C言語ヘッダー生成:     {header_path.name}")

仕様書 Markdown 生成: writer_output_spec.md
C言語ヘッダー生成:     writer_output.h

## 5. `BinaryReader` によるストリーム読み込み検証

In [6]:
reader = BinaryReader(raw_data, default_endian="little")
magic = reader.read_uint32()
ver_maj = reader.read_uint16()
ver_min = reader.read_uint16()
app_name = reader.read_cstring(encoding="utf-8")
doc_title = reader.read_prefixed_string(prefix_bytes=2, encoding="utf-8")
auth_tag = reader.read_fixed_string(length=8, encoding="utf-8").strip()

chunk_type = reader.read_uint16()
payload = reader.read_struct(candidates[chunk_type])

num_recs = reader.read_uint16()
read_records = [reader.read_struct(DataRecord) for _ in range(num_recs)]

print(f"magic:        0x{magic:08X}")
print(f"app_name:     {app_name}")
print(f"chunk_type:   {chunk_type} -> {type(payload).__name__}")
print(f"read_records: {len(read_records)} records")

assert magic == 0x46494C45
assert app_name == "SampleApp v2.0"
assert len(read_records) == 3

magic:        0x46494C45
app_name:     SampleApp v2.0
chunk_type:   1 -> HeaderChunk
read_records: 3 records

## 6. デバッグダンプ (`hexdump`, `dump('table')`)

ライターが追跡した各フィールドのバイト境界を注釈付きで確認できます。

In [7]:
print("--- Annotated Hexdump ---")
print(writer.hexdump()[:500] + "\n  ...")

print("\n--- Field Trace Table ---")
print(writer.dump("table"))

--- Annotated Hexdump ---
Offset    00 01 02 03 04 05 06 07  08 09 0A 0B 0C 0D 0E 0F  |     ASCII      |  Field Annotations
-------------------------------------------------------------------------------------------------
00000000  45 4c 49 46 02 00 00 00  53 61 6d 70 6c 65 41 70  |ELIF....SampleAp|  magic=0x46494C45 (UInt32); version_major=2 (UInt16); version_minor=0 (UInt16); app_name=SampleApp .. (CString)
00000010  70 20 76 32 2e 30 00 15  00 43 6f 6e 66 69 64 65  |p v2.0...Confide|  app_name=SampleApp .. (CString); 
  ...

--- Field Trace Table ---
+--------+------+------------------+-------------------+--------+----------------------------+-----------------+-----------------+
| Offset | Size | Field Name       | Type              | Endian | Hex Bytes                  | Value / Preview | Caption         |
+========+======+==================+===================+========+============================+=================+=================+
| 0x0000 |   4B | magic            | UInt32    

## 7. 先読み (`peek`) と位置保護 (`preserve_position`)

カーソルを進めずに次の値を調べる `peek` や、一時的に別オフセットを読み書きして元の位置へ自動復帰する `preserve_position()` を利用できます。

In [8]:
reader.seek(0)
# 先読み（カーソル位置は 0 のまま進まない）
peeked_magic = reader.peek_uint32()
assert peeked_magic == 0x46494C45
assert reader.tell() == 0

# 一時的な位置移動と自動復帰
with reader.preserve_position():
    reader.seek(8)
    name = reader.read_cstring()
    print(f"preserve_position 内部で読んだ名前: {name}")

# ブロックを抜けると自動でオフセット 0 に復帰！
assert reader.tell() == 0
print("peek と preserve_position の検証成功！")

preserve_position 内部で読んだ名前: SampleApp v2.0
peek と preserve_position の検証成功！

## 8. バイナリ差分比較 (`diff_dump` / `writer.diff()`)

期待値と実際のバイナリに不一致が発生した際、どのバイトがどのように食い違っているかをビジュアルに差分比較できます。

In [9]:
from binary_master import diff_dump

expected = raw_data
# 意図的に 1 バイト改変したデータを作成
corrupted = bytearray(raw_data)
corrupted[2] = 0x99

diff_report = diff_dump(expected[:16], corrupted[:16])
print("バイナリ差分レポート:")
print(diff_report)
print("Procedural Writer の全検証完了！")

バイナリ差分レポート:
--- Binary Diff: Expected vs Actual ---
  Size Expected:  16 bytes (`0x0010`)
  Size Actual: 16 bytes (`0x0010`)
  Differing byte count: 1 bytes in 1 range(s)

Offset      Expected Hex            Actual Hex              Field / Context
---------------------------------------------------------------------------
0x0002..0003   49                      99                      -
Procedural Writer の全検証完了！